# 04 · Federated platinum models → local NEPC / AVPC

Wrapper notebook. All logic lives in `survival_common/federated_endpoint_transfer.py`
(CLI equivalent: `python -m survival_common.federated_endpoint_transfer`).

The federated bundles are XGBoost and elastic-net Cox models trained on **time to platinum**
in the caia-project-compass `rhino_scripts` environments, refit **without Dana-Farber**
(`rhino_results/federated_model_results/federated_*_no_DFCI/`: Fred Hutch, JHU and MSK). This notebook scores
those frozen models on the full local DFCI ADT cohort. No DFCI patient was in their training data, so this is an
external validation on all three endpoints: the fine-grained LLM endpoints **time to NEPC** and **time to AVPC**,
and **time to platinum** (the training endpoint) as a same-endpoint reference.

The pooled preprocessing in each bundle (impute means, centers and scales) is applied as shipped
and never refit locally. Every endpoint shares the same feature frame, so each patient's risk score
is the same for every endpoint; only the outcome changes.

**Metrics** for each endpoint × landmark × bundle × config:
- Mean AUC(t): the IPCW cumulative/dynamic AUC averaged over 1, 2, 3 and 5 years after the landmark (horizons
  without an event or follow-up are dropped), with a patient-bootstrap 95% CI. Censoring is estimated on the local
  cohort. This is the figure metric.
- Paired ΔAUC(t) vs the bundle's age-only `baseline` config.
- Harrell's C and paired ΔC (CSV only), Uno's C, per-horizon AUC(t), and Cox HR per SD of risk score.

The notebook writes `transfer_metrics.csv` and `transfer_auc_t.csv` to `OUTPUT_DIR`
(`rhino_results/federated_transfer_adt_no_DFCI`). After this notebook, rerun `06_prep_figure_data.ipynb`, then
`07_figures.R`, to refresh `federated/transfer_discrimination_no_dfci__platinum` and federated Figure 5.

**Transferred model vs Gleason and TP53/RB1/PTEN** (federated Figure 5 b–d). Panel a is the full ADT cohort above.
The transferred scores are then restricted to three clinical subpopulations at each landmark:
- **b · `gleason`**: patients with a Gleason score (ADT clock). Local Cox: **Gleason (continuous) + age**.
- **c · `trio`**: sequenced patients (sequencing clock). Local Cox: **TP53_ANY + RB1_ANY + PTEN_ANY + age**.
- **d · `trio_gleason`**: sequenced patients with a Gleason score (sequencing clock). Local Cox:
  **Gleason + TP53_ANY + RB1_ANY + PTEN_ANY + age**.

In each, three scores are compared on the same patients: the transferred `both` score, the unpenalized local Cox, and
a combined local Cox of the transferred score (z-scored within the subpopulation) + the same clinical covariates. The
logic lives in `federated_clinical_comparison.py`.
- The Gleason arm uses the ADT clock (landmark = ADT start + landmark). The somatic arms are anchored on the
  sequencing date: the same frozen bundles are re-scored on labs before sequencing + landmark, with outcomes from
  sequencing + landmark (`res_seq`, written under `OUTPUT_DIR/sequencing_anchor/`). The sequencing date is the
  report date of the test reported closest to ADT start. No somatic data is used on the ADT clock.
- Clinical data must be available strictly before the landmark, the same rule the labs follow (the anchoring
  specimen defines time 0 for the somatic arms, so it is always available; Gleason must be available before
  sequencing + landmark).
- Both Cox scores are 5-fold out-of-fold, so no patient is scored by a model fitted on their own outcome.
- Mean AUC(t) uses the horizons estimable in each subpopulation. ΔAUC(t) vs the transferred score (local Cox − labs,
  combined − labs) and combined − local Cox are bootstrapped over the same patients (C and ΔC are kept in the CSV).

This section writes `transfer_clinical_comparison.csv` (the figure input), `transfer_clinical_cox_coefs.csv` and
`transfer_subpop_risk_summary.csv` to `OUTPUT_DIR`.

## Configuration

In [ ]:
import os, sys, json
from pathlib import Path

sys.path.insert(0, ".")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from survival_common.federated_inference import load_bundle
from survival_common import federated_endpoint_transfer as fet
from survival_common import federated_scoring as fs

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT

# >>> Edit: federated delivery, bundles refit without DFCI (Fred Hutch + JHU + MSK) <<<
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(Path(DATA_ROOT) / "rhino_results")))
FEDERATED_RESULTS_DIR = RHINO_ROOT / "federated_model_results"
BUNDLE_GLOB = "federated_*_no_DFCI/*_model_adt.json"   # XGBoost + elastic-net bundles; or list files in BUNDLE_PATHS

DATA_PATH   = Path(DATA_ROOT) / "longitudinal_prediction_data_adt.csv"
OUTPUT_DIR  = RHINO_ROOT / "federated_transfer_adt_no_DFCI"
LAB_MAPPING = fet.DEFAULT_LAB_MAPPING

ENDPOINTS   = ("nepc", "avpc", "platinum")
EXCLUSION   = "none"            # "none" = full ADT cohort; "pre_anchor_castrate" for noprecastrate bundles
LANDMARKS   = None              # None = every landmark present in the bundles
HORIZONS    = fet.DEFAULT_HORIZONS_DAYS   # AUC(t) horizons, days after the landmark
N_BOOTSTRAP = 200
SEED        = 0

# Figure 5 b-c: transferred labs score vs local Gleason / TP53-RB1-PTEN Cox
import federated_clinical_comparison as fcc
GLEASON_PATH          = fcc.sg.DEFAULT_GLEASON_PATH
SOMATIC_PATH          = fcc.sg.DEFAULT_SOMATIC_PATH
SOMATIC_MANIFEST_PATH = fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH
CLINICAL_CV_FOLDS     = 5     # out-of-fold folds for the local Cox
CLINICAL_MIN_EVENTS   = 5     # fewer events in a subpopulation -> count-only row

BUNDLE_PATHS = sorted(FEDERATED_RESULTS_DIR.glob(BUNDLE_GLOB))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(BUNDLE_PATHS)} bundle(s) in {FEDERATED_RESULTS_DIR}")
for p in BUNDLE_PATHS:
    print("  ", p.relative_to(FEDERATED_RESULTS_DIR))

## Load bundles

In [ ]:
def load_bundles(paths):
    """Load bundles keyed by file stem; return (bundles, one row per model)."""
    assert paths, f"no bundles matching {BUNDLE_GLOB!r} in {FEDERATED_RESULTS_DIR}"
    bundles, rows = {}, []
    for path in paths:
        assert path.stem not in bundles, f"duplicate bundle stem {path.stem!r}: {path}"
        b = load_bundle(path)
        bundles[path.stem] = b
        label = str(b.get("analysis_label") or "")
        if ("noprecastrate" in label) != (EXCLUSION == "pre_anchor_castrate"):
            print(f"WARNING: {path.stem} analysis_label={label!r} but EXCLUSION={EXCLUSION!r}")
        for m in b["models"]:
            rows.append({
                "bundle": path.stem, "source": path.parent.name, "model_family": b.get("model_family"),
                "analysis_label": label, "trained_endpoint": b.get("endpoint", "platinum"),
                "landmark_days": m["landmark_days"], "config": m.get("config"),
                "n_covariates": len(m["preprocessing"]["covariate_cols"]),   # XGBoost bundles keep it only here
            })
    return bundles, pd.DataFrame(rows)

bundles, bundle_summary = load_bundles(BUNDLE_PATHS)
bundle_summary

## Local cohort

In [ ]:
local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
lab_mapping = fet.load_lab_mapping(LAB_MAPPING)

pt = local.drop_duplicates(fet.LOCAL_ID_COL)
print(f"{len(local):,} rows, {len(pt):,} patients from {DATA_PATH}")
pd.DataFrame({
    ep: {"events": int(pt[ind].fillna(0).astype(int).sum()),
         "dated events": int((pt[ind].fillna(0).astype(int).eq(1) & pt[dt].notna()).sum())}
    for ep, (ind, dt) in fet.ENDPOINT_SOURCES.items() if ep in ENDPOINTS
}).T

## Run transfer evaluation

In [ ]:
# Shared by the ADT- and sequencing-anchored runs so their bootstrap resamples and horizons match.
TRANSFER_KW = dict(
    endpoints=ENDPOINTS, lab_mapping=lab_mapping, exclusion=EXCLUSION,
    landmarks=LANDMARKS, horizons_days=HORIZONS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED,
)

def write_outputs(res, bundles, paths, out_dir):
    for name in ("scores", "metrics", "auc_t", "coverage", "unit_check"):
        if not res[name].empty:
            res[name].to_csv(out_dir / f"transfer_{name}.csv", index=False)
    with open(out_dir / "transfer_run.json", "w") as fh:
        json.dump({
            "data": str(DATA_PATH),
            "bundles": {p.stem: {"path": str(p), "model_family": bundles[p.stem].get("model_family"),
                                 "analysis_label": bundles[p.stem].get("analysis_label"),
                                 "xgboost_version": bundles[p.stem].get("xgboost_version")}
                        for p in paths},
            "endpoints": list(ENDPOINTS), "exclusion": EXCLUSION,
            "landmarks": list(LANDMARKS) if LANDMARKS else None,
            "horizons_days": list(HORIZONS), "n_bootstrap": N_BOOTSTRAP, "seed": SEED,
            "prep_stats": res["prep_stats"],
        }, fh, indent=2, default=str)
    print(f"wrote outputs -> {out_dir}")

res = fet.evaluate_transfer(local, bundles, **TRANSFER_KW)
write_outputs(res, bundles, BUNDLE_PATHS, OUTPUT_DIR)

In [ ]:
# Trio arm (Figure 5 c) on the sequencing clock: the same frozen bundles re-scored with each sequenced
# patient's sequencing report date (closest to ADT start) as the anchor, so labs come from before sequencing + landmark
# and outcomes run from sequencing + landmark.
clin_gleason, clin_somatic, clin_somatic_features = fcc.load_clinical_sources(
    GLEASON_PATH, SOMATIC_PATH, SOMATIC_MANIFEST_PATH)
specimen = fcc.select_specimen(local, clin_somatic, clin_somatic_features)
local_seq = local
if EXCLUSION != "none":
    # Judge the exclusion on the ADT clock, then re-anchor the patients it keeps.
    kept, _ = fs.apply_exclusion(fet.to_federated_long(local, "platinum", lab_mapping)[0], EXCLUSION)
    local_seq = local[pd.to_numeric(local[fet.LOCAL_ID_COL], errors="coerce").isin(kept[fs.ID_COL].unique())]
local_seq = fcc.sequencing_anchored_local(local_seq, specimen)
seq_offset = specimen[fcc.sg.SEQUENCING_TO_ADT_DAYS]
print(f"{local_seq[fet.LOCAL_ID_COL].nunique():,} sequenced patients; sequencing − ADT start median "
      f"{seq_offset.median():.0f}d (IQR {seq_offset.quantile(.25):.0f} to {seq_offset.quantile(.75):.0f}d), "
      f"{int(seq_offset.lt(0).sum()):,} sequenced before ADT")

SEQ_OUTPUT_DIR = OUTPUT_DIR / "sequencing_anchor"
SEQ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
res_seq = fet.evaluate_transfer(local_seq, bundles, **{**TRANSFER_KW, "exclusion": "none"})
write_outputs(res_seq, bundles, BUNDLE_PATHS, SEQ_OUTPUT_DIR)

## Sanity checks: feature coverage and units

Check these before reading any metric. A covariate the local frame never produces is imputed with the
pooled mean for every patient, which carries no signal. A local mean more than 3× off the pooled mean
usually means a unit mismatch.

The pooled impute means come from Fred Hutch, JHU and MSK only.

In [ ]:
cov = res["coverage"]
display(cov.head())
num = cov.select_dtypes("number").columns.difference(["landmark_days"])
cov.groupby(["bundle", "landmark_days", "config"])[list(num)].mean().round(3)

In [ ]:
uc = res["unit_check"]
flagged = uc[uc["unit_flag"]].sort_values("mean_ratio", ascending=False)
print(f"{flagged['covariate'].nunique()} covariate(s) flagged (>3x from pooled mean)")
flagged.drop_duplicates("covariate")

## Discrimination

In [ ]:
def fmt_ci(est, lo, hi, d=3):
    if not np.isfinite(est):
        return ""
    return f"{est:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]" if np.isfinite(lo) else f"{est:.{d}f}"

def summarize_metrics(metrics):
    """Add formatted CI columns; return (metrics, table of the non-baseline configs)."""
    m = metrics.copy()
    for c in ("c_index_lo", "c_index_hi", "delta_c_vs_baseline", "delta_c_lo", "delta_c_hi",
              "mean_auc_lo", "mean_auc_hi", "delta_auc_vs_baseline", "delta_auc_lo", "delta_auc_hi"):
        if c not in m:
            m[c] = np.nan
    m["C (95% CI)"] = [fmt_ci(r.c_index, r.c_index_lo, r.c_index_hi) for r in m.itertuples()]
    m["ΔC vs age (95% CI)"] = [fmt_ci(r.delta_c_vs_baseline, r.delta_c_lo, r.delta_c_hi) for r in m.itertuples()]
    m["Mean AUC(t) (95% CI)"] = [fmt_ci(r.mean_auc, r.mean_auc_lo, r.mean_auc_hi) for r in m.itertuples()]
    m["ΔAUC vs age (95% CI)"] = [fmt_ci(r.delta_auc_vs_baseline, r.delta_auc_lo, r.delta_auc_hi)
                                 for r in m.itertuples()]
    m["HR/SD (95% CI)"] = [fmt_ci(r.hr_per_sd, r.hr_per_sd_lo, r.hr_per_sd_hi, 2) for r in m.itertuples()]
    summary = m[m["config"] != "baseline"][[
        "endpoint", "landmark_days", "bundle", "n_patients", "n_events",
        "Mean AUC(t) (95% CI)", "ΔAUC vs age (95% CI)", "C (95% CI)", "ΔC vs age (95% CI)",
        "HR/SD (95% CI)", "uno_c",
    ]].round({"uno_c": 3})
    return m, summary.sort_values(["endpoint", "landmark_days", "bundle"]).reset_index(drop=True)

m, summary = summarize_metrics(res["metrics"])
summary

In [ ]:
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # fixed categorical order

def plot_mean_auc(metrics, title, path):
    """Mean AUC(t) (95% bootstrap CI) by landmark; one panel per endpoint, full vs age-only."""
    plot = metrics.dropna(subset=["mean_auc"])
    panels = [ep for ep in ENDPOINTS if ep in set(plot["endpoint"])]
    stems = sorted(plot["bundle"].unique())
    color = {s: COLORS[i % len(COLORS)] for i, s in enumerate(stems)}
    lms = sorted(plot["landmark_days"].unique())
    width = 0.8 / max(1, 2 * len(stems))

    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.6), sharey=True, squeeze=False)
    for ax, ep in zip(axes[0], panels):
        d = plot[plot["endpoint"] == ep]
        k = 0
        for s in stems:
            for cfg, mk, fill in (("both", "o", True), ("baseline", "s", False)):
                r = d[(d["bundle"] == s) & (d["config"] == cfg)].set_index("landmark_days").reindex(lms)
                x = np.arange(len(lms)) + (k - (2 * len(stems) - 1) / 2) * width
                k += 1
                yerr = np.vstack([r["mean_auc"] - r["mean_auc_lo"], r["mean_auc_hi"] - r["mean_auc"]])
                ax.errorbar(x, r["mean_auc"], yerr=yerr, fmt=mk, ms=7, lw=1.5, capsize=0,
                            color=color[s], mfc=color[s] if fill else "white", mew=1.5,
                            label=f"{s} · {'full' if cfg == 'both' else 'age-only'}")
        ax.axhline(0.5, color="#8a8a86", lw=1, ls="--", zorder=0)
        ax.set_xticks(range(len(lms)), [f"{lm}d" for lm in lms])
        n_ev = d.groupby("landmark_days")["n_events"].first().reindex(lms)
        ax.set_title(f"{ep.upper()}  (events: {', '.join(str(int(v)) for v in n_ev.fillna(0))})", fontsize=10)
        ax.set_xlabel("landmark")
        ax.grid(axis="y", color="#e6e5df", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0][0].set_ylabel("Mean AUC(t)")
    axes[0][-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=8)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()

plot_mean_auc(m, "Platinum-trained federated models without DFCI on local endpoints (external)",
              OUTPUT_DIR / "transfer_mean_auc.png")

### Local platinum vs the federated held-out test

`test_c_index` and `n_test` come from the bundle's own pooled 80/20 split of the non-DFCI sites, as recorded in the
export metrics. The local platinum C-index is an external estimate on DFCI patients.

In [ ]:
def metrics_path(bundle_path):
    """Export metrics CSV next to a bundle (…_model_adt.json -> …_metrics_adt.csv)."""
    return bundle_path.with_name(bundle_path.stem.replace("_model_", "_metrics_") + ".csv")

def read_federated_metrics(paths):
    return pd.concat([pd.read_csv(metrics_path(p)).assign(bundle=p.stem)
                      for p in paths if metrics_path(p).exists()], ignore_index=True)

def federated_vs_local(paths, m):
    fed = read_federated_metrics(paths)
    fed = fed[["bundle", "landmark_days", "config", "n_test", "n_events_test", "test_c_index"]]
    loc = m[m["endpoint"] == "platinum"][["bundle", "landmark_days", "config", "n_patients", "n_events", "c_index"]]
    return (fed.merge(loc, on=["bundle", "landmark_days", "config"], how="outer")
        .rename(columns={"test_c_index": "federated_test_C", "c_index": "local_platinum_C",
                         "n_patients": "local_n", "n_events": "local_events"})
        .sort_values(["bundle", "landmark_days", "config"]).round(3).reset_index(drop=True))

federated_vs_local(BUNDLE_PATHS, m)

## Time-dependent AUC

In [ ]:
auc = res["auc_t"]
if auc.empty:
    print("no estimable AUC(t) horizons")
else:
    display(auc.pivot_table(index=["endpoint", "landmark_days", "bundle", "config"],
                            columns="horizon_days", values="auc").round(3))

## Risk-score distributions

The feature frame does not depend on the endpoint, so the score distribution is the same for every
endpoint. The check that matters is whether local scores fall inside the range the pooled model was
fitted on.

In [ ]:
sc = res["scores"]
sc0 = sc[(sc["endpoint"] == ENDPOINTS[0]) & (sc["config"] == "both")]
sc0.groupby(["bundle", "landmark_days"])["risk_score"].describe().round(3)

## Transferred model vs Gleason and TP53/RB1/PTEN

These are the inputs for federated Figure 5 b–d. Each subpopulation is defined per landmark:
- **gleason** (ADT clock): patients with a Gleason score available before TREATMENT_ANCHOR_DATE + landmark. The
  score closest to the landmark is used, and it counts as available from the later of the score date and the date
  of the note that reported it.
- **trio** (sequencing clock): sequenced patients, with the landmark measured from the sequencing date. The
  sequencing date is the report date of the test reported closest to ADT start (same-day reports combined), and
  that specimen is the anchor at every landmark.
  Each gene's `_ANY` indicator is 1 for an SNV, deletion, amplification or SV. Age is age at sequencing.
- **trio_gleason** (sequencing clock): the trio patients who also have a Gleason score available before
  sequencing + landmark.

The transferred scores are the frozen bundles, never refit: `res` (ADT clock) for the Gleason arm and
`res_seq` (sequencing clock) for the trio and trio + Gleason arms. The local and combined (transferred score +
clinical) Cox models are fitted on these patients, so their scores are out-of-fold (stratified on the event). The apparent C and mean AUC(t) are in the CSV for
reference only.

In [ ]:
lms = sorted(res["scores"]["landmark_days"].unique())
# Gleason arm on the ADT clock (Gleason and age only: no somatic data on the ADT clock); trio and
# trio + Gleason arms on the sequencing clock (trio taken from the anchoring specimen, Gleason available
# before sequencing + landmark).
clinical = fcc.build_clinical_features(local, lms, gleason=clin_gleason)
clinical_seq = fcc.build_clinical_features(
    local_seq, lms, gleason=clin_gleason, specimen=specimen, somatic_features=clin_somatic_features)

def membership(scores, clin):
    """Membership among patients actually scored at each landmark (the landmark frame applies the incident gate)."""
    scored_pt = scores.drop_duplicates(["endpoint", "landmark_days", fs.ID_COL])
    member = scored_pt.merge(clin, on=[fs.ID_COL, "landmark_days"], how="left")
    member["any_trio_altered"] = member[list(fcc.TRIO_FEATURES)].eq(1).any(axis=1)
    member["sequenced_with_gleason"] = member[fcc.HAS_SEQUENCING].notna() & member[fcc.GLEASON_FEATURE].notna()
    return member.groupby(["endpoint", "landmark_days"]).agg(
        n_scored=(fs.ID_COL, "size"),
        with_gleason=(fcc.GLEASON_FEATURE, "count"),
        sequenced=(fcc.HAS_SEQUENCING, "count"),
        sequenced_with_gleason=("sequenced_with_gleason", "sum"),
        any_trio_altered=("any_trio_altered", "sum"),
    )

(membership(res["scores"], clinical)[["n_scored", "with_gleason"]]
    .join(membership(res_seq["scores"], clinical_seq)[["sequenced", "sequenced_with_gleason", "any_trio_altered"]]
          .add_suffix(" (sequencing clock)")))

In [ ]:
def clinical_table(comp):
    """One row per endpoint × landmark × subpopulation × bundle: mean AUC(t) of each model and the paired ΔAUC(t)."""
    c = comp.assign(
        AUC=[fmt_ci(r.mean_auc, r.mean_auc_lo, r.mean_auc_hi) for r in comp.itertuples()],
        dAUC=[fmt_ci(r.delta_auc_vs_transferred, r.delta_auc_lo, r.delta_auc_hi) for r in comp.itertuples()],
        dC=[fmt_ci(r.delta_c_vs_transferred, r.delta_lo, r.delta_hi) for r in comp.itertuples()],
        dAUC_clin=[fmt_ci(r.delta_auc_vs_clinical, r.delta_auc_vs_clinical_lo, r.delta_auc_vs_clinical_hi)
                   for r in comp.itertuples()],
    )
    keys = ["endpoint", "landmark_days", "subpopulation", "bundle", "n_scored", "n_scored_events"]
    wide = c.pivot_table(index=keys, columns="model", values="AUC", aggfunc="first")
    wide = wide.reindex(columns=list(fcc.MODELS))
    wide.columns = ["AUC(t) labs (transferred)", "AUC(t) age (transferred)", "AUC(t) local Cox (OOF)",
                    "AUC(t) labs + local Cox (OOF)"]
    cox = c[c["model"] == fcc.CLINICAL_COX].set_index(keys)
    combined = c[c["model"] == fcc.COMBINED_COX].set_index(keys)
    wide["ΔAUC(t) Cox − labs (95% CI)"] = cox["dAUC"]
    wide["ΔAUC(t) labs + Cox − labs (95% CI)"] = combined["dAUC"]
    wide["ΔAUC(t) labs + Cox − Cox (95% CI)"] = combined["dAUC_clin"]
    wide["ΔC Cox − labs (95% CI)"] = cox["dC"]
    return wide.reset_index()

def subpop_summary(scores_adt, scores_seq):
    """Score strata: full cohort and Gleason on the ADT clock, sequencing strata on the sequencing clock."""
    return pd.concat([
        fcc.subpopulation_score_summary(scores_adt, clinical, sections=("cohort", "gleason"))
            .assign(time_origin="ADT start"),
        fcc.subpopulation_score_summary(scores_seq, clinical_seq, sections=("sequencing",))
            .assign(time_origin="sequencing date"),
    ], ignore_index=True)

comp, coefs = fcc.compare_by_subpopulation(
    {"gleason": (res["scores"], clinical), "trio": (res_seq["scores"], clinical_seq),
     "trio_gleason": (res_seq["scores"], clinical_seq)},
    n_folds=CLINICAL_CV_FOLDS, n_bootstrap=N_BOOTSTRAP,
    seed=SEED, min_events=CLINICAL_MIN_EVENTS, horizons_days=HORIZONS)
subpop = subpop_summary(res["scores"], res_seq["scores"])
fcc.write_outputs(comp, coefs, subpop, OUTPUT_DIR)

fit_issues = comp[(comp["n_failed_folds"] > 0) | (comp["n_convergence_warnings"] > 0) | comp["note"].ne("")]
if len(fit_issues):
    display(fit_issues.drop_duplicates(["endpoint", "landmark_days", "subpopulation"])[
        ["endpoint", "landmark_days", "subpopulation", "n_patients", "n_events", "n_folds",
         "n_failed_folds", "n_convergence_warnings", "note"]])
clinical_table(comp)

In [ ]:
# Local Cox fitted on the whole subpopulation (apparent fit; HR per unit, age per year)
display(coefs.assign(**{"HR (95% CI)": [fmt_ci(r.hr, r.hr_lo, r.hr_hi, 2) for r in coefs.itertuples()]})
        .pivot_table(index=["subpopulation", "covariate"], columns=["endpoint", "landmark_days"],
                     values="HR (95% CI)", aggfunc="first"))

# Transferred labs score inside the subpopulations (pooled over endpoints; the score does not depend on them)
subpop.pivot_table(index=["landmark_days", "stratum"], columns="bundle", values="median", sort=False).round(3)

In [ ]:
fcc.plot_clinical_comparison(comp, OUTPUT_DIR / "transfer_clinical_comparison.png", ENDPOINTS)